# DRF ViewSets & Routers

## From Two Views to One ViewSet

With generic views, a typical resource requires two classes:
```python
class BookListView(ListCreateAPIView): ...
class BookDetailView(RetrieveUpdateDestroyAPIView): ...
```

A `ViewSet` collapses these into one class.


## ViewSet (Manual)

```python
from rest_framework import viewsets
from rest_framework.response import Response
from .models import Book
from .serializers import BookSerializer

class BookViewSet(viewsets.ViewSet):
    def list(self, request):
        qs = Book.objects.all()
        return Response(BookSerializer(qs, many=True).data)

    def retrieve(self, request, pk=None):
        book = Book.objects.get(pk=pk)
        return Response(BookSerializer(book).data)
```

When not using a router, wire the action dict manually:
```python
book_list = BookViewSet.as_view({'get': 'list', 'post': 'create'})
book_detail = BookViewSet.as_view({'get': 'retrieve', 'put': 'update', 'delete': 'destroy'})

urlpatterns = [
    path('books/', book_list),
    path('books/<int:pk>/', book_detail),
]
```


## ModelViewSet

`ModelViewSet` provides all five CRUD actions automatically:

```python
class BookViewSet(viewsets.ModelViewSet):
    queryset = Book.objects.all()
    serializer_class = BookSerializer
```

This gives `list`, `retrieve`, `create`, `update`, and `destroy` with no extra code.


## Routers: SimpleRouter and DefaultRouter

Routers automatically generate URL patterns for a registered ViewSet.

```python
from rest_framework.routers import DefaultRouter
from .views import BookViewSet

router = DefaultRouter()
router.register('books', BookViewSet, basename='book')
urlpatterns = router.urls
```

**Generated routes:**
- `GET /books/` — list
- `POST /books/` — create
- `GET /books/<pk>/` — retrieve
- `PUT/PATCH /books/<pk>/` — update
- `DELETE /books/<pk>/` — destroy

**DefaultRouter vs SimpleRouter:**
- `DefaultRouter` adds a root API view at `/` listing all registered endpoints.
- Both support `?format=json` and `.json` format suffixes.


## Custom Actions with @action

```python
from rest_framework.decorators import action
from rest_framework.permissions import IsAuthenticated

class BookViewSet(viewsets.ModelViewSet):
    queryset = Book.objects.all()
    serializer_class = BookSerializer

    # Collection-level: GET /books/recent/
    @action(detail=False, methods=['get'])
    def recent(self, request):
        qs = self.get_queryset().order_by('-id')[:5]
        return Response(self.get_serializer(qs, many=True).data)

    # Detail-level: POST /books/<pk>/publish/
    @action(detail=True, methods=['post'], url_path='publish',
            permission_classes=[IsAuthenticated])
    def publish_book(self, request, pk=None):
        book = self.get_object()
        return Response({"status": f"Book {book.id} published"})
```

- `detail=False` → collection URL (`/books/recent/`)
- `detail=True` → detail URL (`/books/<pk>/publish/`)
- Use `url_path` to customize the URL segment.


## Serializer Relationships

### PrimaryKeyRelatedField (default — integer IDs)
```python
class BookSerializer(serializers.ModelSerializer):
    author = serializers.PrimaryKeyRelatedField(queryset=Author.objects.all())
    class Meta:
        model = Book
        fields = ['id', 'title', 'author']
```

### SlugRelatedField (human-readable slugs)
```python
class BookSerializer(serializers.ModelSerializer):
    author = serializers.SlugRelatedField(slug_field='slug', queryset=Author.objects.all())
    class Meta:
        model = Book
        fields = ['id', 'title', 'author']
```

### Nested Read-Only + Slug Writes (recommended pattern)
```python
class BookSerializer(serializers.ModelSerializer):
    author = serializers.StringRelatedField(read_only=True)
    author_slug = serializers.SlugRelatedField(
        slug_field='slug', source='author', queryset=Author.objects.all(), write_only=True
    )
    class Meta:
        model = Book
        fields = ['id', 'title', 'author', 'author_slug']
```


## depth, HyperlinkedModelSerializer, and Validation

### depth
Auto-expands related models (read-only only):
```python
class BookSerializer(serializers.ModelSerializer):
    class Meta:
        model = Book
        fields = ['id', 'title', 'author']
        depth = 1
```

### HyperlinkedModelSerializer
Clients send and receive full URLs instead of IDs:
```python
class BookSerializer(serializers.HyperlinkedModelSerializer):
    class Meta:
        model = Book
        fields = ['url', 'id', 'title', 'author']
        extra_kwargs = {'url': {'view_name': 'book-detail'}}
```

### Field Validation
```python
def validate_title(self, value):
    if 'django' not in value.lower():
        raise serializers.ValidationError("Title must include 'django'.")
    return value
```


## Summary

- `ViewSet` groups all actions for a resource into one class.
- `ModelViewSet` auto-provides the full CRUD action set.
- `DefaultRouter` generates all URL patterns from a single `router.register()` call.
- `@action` adds custom endpoints at collection or detail level without extra URL code.
- Use nested read-only serializers for output and slug/PK fields for write — this is the most practical daily pattern.
- `depth = 1` auto-expands related objects for read-only responses.
- `HyperlinkedModelSerializer` uses full URLs as identifiers.
